# DistilBERT

**Goal**

1. Imports                         
2. Check PyTorch                   
3. Load tokenizer
4. Tokenization
5. Load pretrained DistilBERT
6. Inspect hidden representations
7. Load classification model
8. Test logits

**Focal action**

        Yelp dataset
            ↓
        subset
            ↓
        tokenization
            ↓
        fine-tuning
            ↓
        evaluation

In [22]:
pip install transformers torch accelerate

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip show transformers
!pip show torch

Name: transformers
Version: 5.17.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\Francesco\Documents\AI\Portfolio\.venv\Lib\site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: 
Name: torch
Version: 2.14.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT


## 1. Import

In [2]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
Transformers: 5.17.0
CUDA available: False


`CUDA available: False`

This means that your machine learning library (such as PyTorch) is unable to detect or use the NVIDIA graphics card (GPU) to accelerate computations. As a result, any model or script will run on the CPU, making training or data processing much slower.

**Baseline vs DistilBERT**

    REVIEW
      │
      ▼
    TF-IDF
      │
      ▼
    2M+ features
      │
      ▼
    Logistic Regression
      │
      ▼
    sentiment

vs

    REVIEW
      │
      ▼
    TOKENIZER
      │
      ▼
    TOKENS
      │
      ▼
    DISTILBERT
      │
      ▼
    REPRESENTATION
      │
      ▼
    CLASSIFIER
      │
      ▼
    sentiment



**Tokenizer**

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

**Check a review**

In [13]:
text = "The food was absolutely amazing!"

tokens = tokenizer.tokenize(text)

print(tokens)

['the', 'food', 'was', 'absolutely', 'amazing', '!']


In [14]:
token_ids = tokenizer.encode(text)

print(token_ids)

[101, 1996, 2833, 2001, 7078, 6429, 999, 102]


**[CLS], [SEP] e token speciali**

In [15]:
encoded = tokenizer(
    text,
    return_tensors="pt"
)

print(encoded)

{'input_ids': tensor([[ 101, 1996, 2833, 2001, 7078, 6429,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [16]:
print(
    tokenizer.convert_ids_to_tokens(
        encoded["input_ids"][0]
    )
)

['[CLS]', 'the', 'food', 'was', 'absolutely', 'amazing', '!', '[SEP]']


## 2. Check PyTorch

**Test the pre-trained model**

In [21]:
from transformers import AutoModel

model_bert = AutoModel.from_pretrained(
    "distilbert-base-uncased"
)

with torch.no_grad():
    outputs = model_bert(**encoded)

print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 8, 768])


DistilBERT:

`torch.Size([1, 8, 768])` 

    1 → numero di frasi
    8 → numero di token
    768 → dimensione della rappresentazione

TF-IDF produceed instead:

        1 review
           ↓
        2,049,634 features


**Sentiment classification**

In [ ]:
from transformers import AutoModelForSequenceClassification

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Inference test

In [19]:
inputs = tokenizer(
    "The food was absolutely amazing!",
    return_tensors="pt"
)

with torch.no_grad():
    outputs = sentiment_model(**inputs)

print(outputs.logits)

tensor([[0.0840, 0.2148]])


`tensor([[0.0840, 0.2148]])` 

The two numbers are the logits of the classes  
- class 0
- class 1

** Conversion in probability

In [20]:
probabilities = torch.softmax(
    outputs.logits,
    dim=-1
)

print(probabilities)

tensor([[0.4673, 0.5327]])


**Dataset ?**

The dataset cannot be made now due to there are:
- 560,000 training reviews
- 38,000 test reviews

Fine-tuning of DistilBERT on 560k reviews can be very heavy on the CPU.

Step A

      Tokenizer
         ↓
      Pretrained DistilBERT
         ↓
      Understand the inference


Step B

      Small subset
         ↓
      Fine-tuning
         ↓
      Validation


Step C

      Evaluation
         ↓
      Compare TF-IDF vs DistilBERT

## 3. Load tokenizer 

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Esample:

        "The food was absolutely amazing!"
                   ↓
        ["the", "food", "was", "absolutely", "amazing", "!"]
                   ↓
        [1996, 2833, 2001, 7078, 6429, 999]

In [25]:
print(tokenizer)
print("Vocabulary size:", tokenizer.vocab_size)

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
Vocabulary size: 30522


## 4. Tokenization

**Tokens**

In [ ]:
text = "The food was absolutely amazing!"
tokens = tokenizer.tokenize(text)
print(tokens)

token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

encoded = tokenizer(text)
print(encoded)

['the', 'food', 'was', 'absolutely', 'amazing', '!']
[1996, 2833, 2001, 7078, 6429, 999]
{'input_ids': [101, 1996, 2833, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


`attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]`

all of the 8 tokens are actual tokens. With longer phrases and padding application, zeros will occur

**Special tokens**

In [31]:
encoded = tokenizer(text, return_tensors="pt")

print(encoded)
print()
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

{'input_ids': tensor([[ 101, 1996, 2833, 2001, 7078, 6429,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

['[CLS]', 'the', 'food', 'was', 'absolutely', 'amazing', '!', '[SEP]']


[CLS]
[SEP]

are special tokens used by the model for limiting and representing the sequence.

`return_tensors="pt"`

Return data as PyTorch tensor

In [32]:
print(type(encoded["input_ids"]))
print(encoded["input_ids"].shape)

<class 'torch.Tensor'>
torch.Size([1, 8])


1 phrase, 8 tokens

## 5. Load pretrained DistilBERT

In [34]:
from transformers import AutoModel
model_bert = AutoModel.from_pretrained("distilbert-base-uncased")

print(model_bert)

with torch.no_grad():
    outputs = model_bert(**encoded)

print(outputs)
print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

        [1, 8, 768]
         │  │   │
         │  │   └── 768 numbers for each token
         │  └────── 8 token
         └───────── 1 phrases

In [35]:
from transformers import AutoModel

model_bert = AutoModel.from_pretrained("distilbert-base-uncased")

with torch.no_grad():
    outputs = model_bert(**encoded)

print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 8, 768])


The original checkpoint also contains some weights related to the language modeling component (vocab_*) that are not needed by the DistilBertModel we loaded.

## 6. Inspect hidden representations

In [36]:
outputs.last_hidden_state

hidden_states = outputs.last_hidden_state
print(hidden_states.shape)

torch.Size([1, 8, 768])


Taking the first token "CLS"

In [39]:
cls_embedding = hidden_states[0, 0]
print(cls_embedding.shape)
print(cls_embedding[:10])

torch.Size([768])
tensor([-0.0611,  0.0459,  0.1065,  0.0170,  0.0580, -0.2162,  0.1429,  0.5789,
        -0.0985,  0.0224])


These numbers are the contextual vector representation of the [CLS] token.

Understanding the context with an exampple:
`“The food was absolutely amazing”`

The model does not simply represent “amazing” with a fixed vector.
The representation of “amazing” is calculated by taking into account the other tokens in the sentence.

        amazing
           ↓
        "amazing" + phrase context
           ↓
        vector of 768 dimensions

**TF-IDF**

        "The food was absolutely amazing"
             ↓
        TF-IDF
             ↓
        huge, sparse vector
        2049634 dimensions

The model primarily takes into account the presence and weight of words.

**DistilBERT**

        "The food was absolutely amazing"
             ↓
        Transformer
             ↓
        8 token × 768 valori

The representations are contextual.


**Checking all the tokens**

In [1]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

for token, embedding in zip(tokens, hidden_states[0]):
    print(token, embedding.shape)

NameError: name 'tokenizer' is not defined

These numbers are **NOT yet “sentiment”**

If ww print `cls_embedding[:10]` and see: `tensor([...])`

we cannot say:

*“The first number indicates how positive the review is.”*

These 768 values are latent features learned during pre-training.

The model has learned to represent linguistic and semantic aspects of the text.

For example, during pre-training, it may have learned representations useful for:

- relationships between words
- sentence structure
- context
- meaning
- dependencies between tokens

But we haven’t trained DistilBERT on our *Yelp* problem yet.

**Representations comparison**

*"The food was absolutely **amazing**!"*

*"The food was absolutely **terrible**!"*

In [ ]:
text_positive = "The food was absolutely amazing!"
text_negative = "The food was absolutely terrible!"

encoded_positive = tokenizer(text_positive, return_tensors="pt")
encoded_negative = tokenizer(text_negative, return_tensors="pt")

with torch.no_grad():
    output_positive = model_bert(**encoded_positive)
    output_negative = model_bert(**encoded_negative)

cls_positive = output_positive.last_hidden_state[0, 0]
cls_negative = output_negative.last_hidden_state[0, 0]

print("Positive:", cls_positive[:10])
print("Negative:", cls_negative[:10])